# Notebook 2 - Data Inspection
Sprint 4 - EDA

In Notebook 1 I understood what my dataset is about. In this notebook I inspect it properly: size, column names, data types, a look at the actual rows, summary statistics, unique values and value counts. At the end I identify which columns are numerical, categorical, date/time, identifier and which can be the target.

**Dataset:** Telco Customer Churn (Kaggle, 7043 rows and 21 columns)

In [18]:
import pandas as pd
import numpy as np

df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")
print("Loaded")

Loaded


## 1. Number of Rows, Number of Columns and shape

**Concept:** `df.shape` returns a tuple (rows, columns). Rows are the records and columns are the variables. `len(df)` also gives the number of rows and `df.columns` gives the column labels.

**Example (numerical):** A table of 30 students with Name, Age, Marks and City has shape (30, 4). If I later add 5 new students and a column for Gender, the shape becomes (35, 5). This shows how shape changes whenever rows or columns are added or removed, so it is the first thing to check after every step.

In [19]:
print("Shape:", df.shape)
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])
print("len(df):", len(df))

Shape: (7043, 21)
Number of rows: 7043
Number of columns: 21
len(df): 7043


**What the code does:** `shape` gives the pair (rows, columns). I print the two values separately with `shape[0]` and `shape[1]` and also use `len(df)` to confirm the row count.

**Why this method was used:** It is the quickest way to know the size of the data, and it needs no extra calculation.

**What the output means:** The dataset has 7043 rows (customers) and 21 columns (details about each customer).

**Insights:** The size is small enough to work on a normal laptop, and there are many rows compared to the number of columns, which is good for learning patterns.

**Effect on the ML pipeline:** I will compare the shape again after cleaning (for example after removing duplicates or dropping columns) to make sure I did not lose rows by mistake. Also, model choice depends on data size.

## 2. Column Names and Index

**Concept:** `df.columns` lists the column names and `df.index` shows the row labels. By default pandas gives a `RangeIndex` that starts from 0.

**Example (business):** In a bank table the index could be the account number, so that we can find a customer directly using `df.loc[account_number]`. In my dataset there is no useful label like that, so I keep the default 0 to 7042 index and keep `customerID` as a normal column.

In [20]:
print(df.columns.tolist())
print()
print(df.index)
print()
print("Is customerID unique?", df["customerID"].is_unique)

['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']

RangeIndex(start=0, stop=7043, step=1)

Is customerID unique? True


**What the code does:** `df.columns.tolist()` prints all column names as a list and `df.index` prints the type and range of the row labels. I also checked if `customerID` is unique.

**Why this method was used:** Column names are needed every time I select or drop a column, and I want to spot spelling or case problems early. The index tells how rows are identified.

**What the output means:** There are 21 column names. The index is a RangeIndex from 0 to 7042, which means pandas just numbered the rows. `customerID` is unique.

**Insights:** Column names are clean (no spaces) but the style is mixed, for example `tenure` is lowercase while `Contract` starts with a capital letter. Since `customerID` is unique I could use it as an index, but I am keeping it as a column for now.

**Effect on the ML pipeline:** Consistent column names avoid errors in code. The index should not become a feature by mistake, and `customerID` has to be removed from the training features later.

## 3. Data Types (dtypes)

**Concept:** Each column has a data type, such as `int64`, `float64` or `object` (text). The type decides what operations are possible.

**Example (AI/ML):** Suppose a column with prices is stored as text like "1,200". The model cannot do maths on text, so I would first have to convert it to numbers. Wrong data types are one of the most common reasons a pipeline fails.

In [21]:
print(df.dtypes)
print()
print(df.dtypes.value_counts())

customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges            str
Churn                   str
dtype: object

str        18
int64       2
float64     1
Name: count, dtype: int64


**What the code does:** `df.dtypes` shows the type of every column and `value_counts()` on it counts how many columns have each type.

**Why this method was used:** I need to check whether the stored type matches what the column really means.

**What the output means:** There are 2 int-type columns (`SeniorCitizen`, `tenure`), 1 float column (`MonthlyCharges`) and 18 object columns.

**Insights:** `TotalCharges` is object although it holds money amounts, so something is wrong in that column. `SeniorCitizen` is int but it is a Yes/No (0/1) column, so it is really categorical.

**Effect on the ML pipeline:** `TotalCharges` has to be converted to numeric before scaling or modelling. `SeniorCitizen` should be treated as a category. I will fix these in the preprocessing sprint.

## 4. head(), tail() and sample()

**Concept:**
- `head()` shows the first rows.
- `tail()` shows the last rows.
- `sample()` shows random rows.

**Example (business):** If a sales file is sorted by date, `head()` shows the oldest orders and `tail()` the newest. If I only look at `head()` I may miss problems or a different pattern at the end of the file, and `sample()` gives a random mix, so I get a fairer picture.

In [22]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [23]:
df.tail()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
7038,6840-RESVB,Male,0,Yes,Yes,24,Yes,Yes,DSL,Yes,...,Yes,Yes,Yes,Yes,One year,Yes,Mailed check,84.80,1990.5,No
7039,2234-XADUH,Female,0,Yes,Yes,72,Yes,Yes,Fiber optic,No,...,Yes,No,Yes,Yes,One year,Yes,Credit card (automatic),103.20,7362.9,No
7040,4801-JZAZL,Female,0,Yes,Yes,11,No,No phone service,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.60,346.45,No
7041,8361-LTMKD,Male,1,Yes,No,4,Yes,Yes,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Mailed check,74.40,306.6,Yes
7042,3186-AJIEK,Male,0,No,No,66,Yes,No,Fiber optic,Yes,...,Yes,Yes,Yes,Yes,Two year,Yes,Bank transfer (automatic),105.65,6844.5,No


In [24]:
df.sample(5, random_state=42)

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
185,1024-GUALD,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,24.80,24.8,Yes
2715,0484-JPBRU,Male,0,No,No,41,Yes,Yes,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Month-to-month,Yes,Bank transfer (automatic),25.25,996.45,No
3825,3620-EHIMZ,Female,0,Yes,Yes,52,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,19.35,1031.7,No
1807,6910-HADCM,Female,0,No,No,1,Yes,No,Fiber optic,No,...,Yes,No,No,No,Month-to-month,No,Electronic check,76.35,76.35,Yes
132,8587-XYZSF,Male,0,No,No,67,Yes,No,DSL,No,...,No,Yes,No,No,Two year,No,Bank transfer (automatic),50.55,3260.1,No


**What the code does:** `head()` prints the first 5 rows, `tail()` the last 5 rows and `sample(5, random_state=42)` prints 5 random rows. `random_state=42` makes the random rows the same every time I run it.

**Why this method was used:** Looking at real rows helps me see how the values actually look (spelling, formats, blanks), which statistics alone cannot show. Checking the start, the end and a random sample reduces the chance of missing something.

**What the output means:** Each row is one customer. Values like Yes/No, No internet service, Month-to-month and Electronic check are text labels, and `tenure`, `MonthlyCharges` are numbers.

**Insights:** Many columns use the same labels (Yes/No/No internet service). The rows do not look sorted by anything. The data looks consistent in all three views, so I did not see any obvious problem in these rows.

**Effect on the ML pipeline:** Since the categorical columns contain text, they need encoding. The labels like 'No internet service' and 'No phone service' have to be handled carefully because they are a third value in columns that look like Yes/No.

## 5. info()

**Concept:** `info()` prints a compact summary: number of rows, each column name, the number of non-null values, the data type, and the memory used.

**Example (numerical):** If a table has 100 rows and a column shows 92 non-null values, then 8 values are missing in that column. `info()` gives this count for every column in one shot.

In [25]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

**What the code does:** `info()` prints the row count, the non-null count and dtype of each column, and the memory used by the DataFrame.

**Why this method was used:** It is the fastest way to see missing values and data types together.

**What the output means:** All 21 columns show 7043 non-null values, so pandas does not report any missing values. 18 columns are object, which includes `TotalCharges`.

**Insights:** It looks like there are no missing values, but this is misleading. `TotalCharges` has blank spaces (`' '`) in 11 rows, and pandas does not count a blank space as null. I will confirm this below and study it in the Data Quality notebook.

**Effect on the ML pipeline:** If I trusted only `info()` I would think no missing value treatment is needed and the model would break later on `TotalCharges`. So `isnull()` and `info()` are not enough, hidden missing values must be checked separately.

In [26]:
# checking the TotalCharges problem I noticed
print(df["TotalCharges"].isnull().sum())
print(df["TotalCharges"].value_counts().head(3))

0
TotalCharges
20.2     11
         11
19.75     9
Name: count, dtype: int64


The most repeated value in `TotalCharges` is a blank space (11 times), while `isnull()` shows 0. This confirms the hidden missing values.

## 6. describe()

**Concept:** `describe()` gives summary statistics for numerical columns: count, mean, standard deviation, minimum, 25%, 50% (median), 75% and maximum. With `include="object"` it gives count, unique, top value and its frequency for text columns.

**Example (business):** For monthly bills, if the mean is 65 and the median is 70, then most customers pay close to 70 and a few low bills pull the average down. Comparing the mean and the median quickly tells whether the data is skewed.

In [27]:
df.describe()

,SeniorCitizen,tenure,MonthlyCharges
count,7043.000000,7043.000000,7043.000000
mean,0.162147,32.371149,64.761692
std,0.368612,24.559481,30.090047
min,0.000000,0.000000,18.250000
25%,0.000000,9.000000,35.500000
50%,0.000000,29.000000,70.350000
75%,0.000000,55.000000,89.850000
max,1.000000,72.000000,118.750000


In [28]:
df.describe(exclude="number").T

,count,unique,top,freq
customerID,7043,7043,7590-VHVEG,1
gender,7043,2,Male,3555
Partner,7043,2,No,3641
Dependents,7043,2,No,4933
PhoneService,7043,2,Yes,6361
MultipleLines,7043,3,No,3390
InternetService,7043,3,Fiber optic,3096
OnlineSecurity,7043,3,No,3498
OnlineBackup,7043,3,No,3088
DeviceProtection,7043,3,No,3095


**What the code does:** `describe()` calculates summary statistics for numeric columns, and `describe(include='object')` gives count, number of unique values, most frequent value (top) and its frequency for text columns.

**Why this method was used:** It gives a quick statistical picture of every column without any plotting, and helps to find strange minimum or maximum values.

**What the output means:** For `tenure` the values go from 0 to 72 months with a mean of about 32 and a median of 29. For `MonthlyCharges` the minimum is about 18 and the maximum is about 119 with a mean of about 65. For `SeniorCitizen` the mean is about 0.16, which means around 16% of customers are senior citizens. `TotalCharges` is missing from the numeric table because it is stored as text.

**Insights:** No negative or impossible values are visible in the numeric columns. The mean of `tenure` is slightly higher than the median, so it may be a little right-skewed. `MonthlyCharges` has a median higher than the mean, so it may be left-skewed. I will check these properly in the distribution notebook. A `tenure` minimum of 0 matches the 11 new customers found earlier.

**Effect on the ML pipeline:** The columns have very different ranges (`tenure` 0-72, `MonthlyCharges` 18-119, and `TotalCharges` is even larger), so scaling will be needed for distance-based models like KNN or SVM. The skew check will tell if transformations are needed.

## 7. nunique()

**Concept:** `nunique()` counts how many different values each column has. It is useful to separate identifier columns, categorical columns and continuous columns.

**Example (AI/ML):** A column with only 2 unique values (Yes/No) is a binary category and can be label encoded. A column with 7043 unique values in 7043 rows is an id and has no use for prediction. A column with 1500 unique city names is high-cardinality, and one-hot encoding it would create 1500 new columns, so a different encoding is needed.

In [29]:
df.nunique().sort_values()

gender                 2
SeniorCitizen          2
Partner                2
Dependents             2
PhoneService           2
PaperlessBilling       2
Churn                  2
MultipleLines          3
TechSupport            3
StreamingTV            3
OnlineBackup           3
DeviceProtection       3
StreamingMovies        3
Contract               3
OnlineSecurity         3
InternetService        3
PaymentMethod          4
tenure                73
MonthlyCharges      1585
TotalCharges        6531
customerID          7043
dtype: int64

**What the code does:** `nunique()` counts the distinct values in every column and `sort_values()` arranges them from smallest to largest.

**Why this method was used:** The number of unique values quickly shows what kind of column it is, without reading the data.

**What the output means:** Many columns have only 2 to 4 unique values (Yes/No columns, `Contract`, `InternetService`, `PaymentMethod`). `tenure` has 73 unique values. `MonthlyCharges` has about 1585 and `TotalCharges` about 6531 unique values. `customerID` has 7043.

**Insights:** Most features are low-cardinality categories. `customerID` is fully unique, so it is an identifier. `tenure` has few unique values compared to the charges columns because it is a count of months. The blank space in `TotalCharges` is counted as one of its unique values, which is another sign of the hidden problem.

**Effect on the ML pipeline:** Low-cardinality columns are safe for one-hot encoding. `customerID` must be dropped. Continuous columns will need scaling.

## 8. value_counts()

**Concept:** `value_counts()` counts how many times each value appears in a column. With `normalize=True` it gives proportions instead of counts.

**Example (business):** If a shop has 100 orders and `value_counts()` on the payment type shows UPI 70, Card 20, Cash 10, the business immediately knows UPI is the main payment mode. The same idea applied to my target `Churn` tells how many customers stayed and how many left.

In [30]:
print(df["Churn"].value_counts())
print()
print(df["Churn"].value_counts(normalize=True).round(3))

Churn
No     5174
Yes    1869
Name: count, dtype: int64

Churn
No     0.735
Yes    0.265
Name: proportion, dtype: float64


In [31]:
for col in ["Contract", "InternetService", "PaymentMethod", "gender"]:
    print(col)
    print(df[col].value_counts())
    print()

Contract
Contract
Month-to-month    3875
Two year          1695
One year          1473
Name: count, dtype: int64

InternetService
InternetService
Fiber optic    3096
DSL            2421
No             1526
Name: count, dtype: int64

PaymentMethod
PaymentMethod
Electronic check             2365
Mailed check                 1612
Bank transfer (automatic)    1544
Credit card (automatic)      1522
Name: count, dtype: int64

gender
gender
Male      3555
Female    3488
Name: count, dtype: int64



**What the code does:** `value_counts()` counts each category in a column and `normalize=True` converts the counts to fractions. I used it on `Churn` and on four important categorical columns in a loop.

**Why this method was used:** It is the standard way to see the distribution of a categorical column and to find rare or unexpected labels.

**What the output means:** About 5174 customers did not churn and 1869 churned, which is roughly 73.5% and 26.5%. Most customers are on a month-to-month contract (3875), then two-year (1695) and one-year (1473). Fiber optic is the most common internet service (3096), followed by DSL (2421) and No (1526). Electronic check is the most common payment method. `gender` is almost equally split between Male and Female.

**Insights:** The target is imbalanced. Month-to-month is the biggest contract group, and I want to check later if it has more churn than the others. There are no strange spellings or rare labels in the columns I checked, so the categories look clean.

**Effect on the ML pipeline:** Because of the imbalance, accuracy will be a misleading metric, and I may need class weights or resampling. The contract and payment groups look like useful features to keep and to encode.

## 9. Identifying column types

Using everything above I now classify the columns.

**Concept:** Before preprocessing, each column should be tagged as numerical, categorical, date/time, identifier or target, because each type is treated differently.

**Example (AI/ML):** In a house price dataset, `Area` is numerical (scale it), `City` is categorical (encode it), `HouseID` is an identifier (drop it), `SaleDate` is date/time (extract year and month), and `Price` is the target. I do the same tagging for the churn data.

In [32]:
# numerical (by meaning)
numerical_cols = ["tenure", "MonthlyCharges", "TotalCharges"]

# identifier
id_cols = ["customerID"]

# target
target_col = "Churn"

# categorical = everything else
categorical_cols = [c for c in df.columns
                    if c not in numerical_cols + id_cols + [target_col]]

# date/time columns - check if there is any
datetime_cols = df.select_dtypes(include=["datetime", "datetimetz"]).columns.tolist()

print("Numerical  :", numerical_cols)
print("Categorical:", len(categorical_cols), "->", categorical_cols)
print("Date/time  :", datetime_cols)
print("Identifier :", id_cols)
print("Target     :", target_col)

Numerical  : ['tenure', 'MonthlyCharges', 'TotalCharges']
Categorical: 16 -> ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']
Date/time  : []
Identifier : ['customerID']
Target     : Churn


In [33]:
# any date-like column name?
print([c for c in df.columns if "date" in c.lower() or "time" in c.lower()])

[]


**What the code does:** I put the columns into lists by their meaning: numerical, categorical, identifier and target. For date/time I used `select_dtypes` and also searched the column names for 'date' or 'time'.

**Why this method was used:** A pandas dtype is not always the real type, so I classified by meaning. For example, `TotalCharges` is numerical although pandas shows object.

**What the output means:** There are 3 numerical columns, 16 categorical columns (including `SeniorCitizen`), 1 identifier (`customerID`) and 1 target (`Churn`). There is no date/time column.

**Insights:** Potential target: `Churn` is the target for classification. `MonthlyCharges` or `TotalCharges` could be targets for a regression problem, but churn is the business problem here. There is no date column, so no time-series analysis is possible, but `tenure` gives some information about time.

**Effect on the ML pipeline:** Each group gets its own preprocessing: scaling for numerical, encoding for categorical, dropping for the identifier, and `Churn` is separated as y. The 11 blank `TotalCharges` values and the `SeniorCitizen` type must be fixed before this split.

## Summary of Data Inspection

| Check | Result |
|---|---|
| Shape | 7043 rows, 21 columns |
| Index | Default RangeIndex 0-7042 |
| Data types | 18 object, 2 int, 1 float |
| Missing values (from info) | None shown, but `TotalCharges` has 11 hidden blank values |
| Date/time columns | None |
| Identifier | `customerID` |
| Target | `Churn` (about 73.5% No, 26.5% Yes) |
| Numerical | `tenure`, `MonthlyCharges`, `TotalCharges` |
| Categorical | 16 columns |

**Problems found for the preprocessing sprint**
1. `TotalCharges` is object type and has 11 blank values.
2. `SeniorCitizen` is int but categorical.
3. Target classes are imbalanced.
4. Numerical columns have different ranges, so scaling is needed.
5. Some columns have a third label like 'No internet service' that looks like a Yes/No column.

**What I learned:** `info()` and `isnull()` alone can say there are no missing values while a column still has hidden blanks, so I must also look at the actual values with `value_counts()` and `head()`.

Next: `03_Data_Quality.ipynb`